<a href="https://colab.research.google.com/github/Djerad/Tranformers/blob/main/Sentiment_Analysis_with_Bert_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Sentiment Analysis with BERT (Fine-tuned on IMDb)
Fixed & Improved Version — all bugs corrected


In [1]:
# ✅ STEP 1: Install dependencies
!pip install transformers datasets torch scikit-learn --quiet


In [2]:
# ✅ STEP 2: Load IMDB dataset
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [3]:
# ✅ STEP 3: Tokenize
# FIX: max_length=128 instead of 512 — faster training, still good accuracy for sentiment
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128  # FIX: was 512 — unnecessary for sentiment, much slower
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Tokenization complete!")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenization complete!


In [4]:

from transformers import AutoModelForSequenceClassification

id2label = {0: "Negative", 1: "Positive"}
label2id = {"Negative": 0, "Positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)
print("Model loaded!")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!


In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}


In [6]:

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    eval_strategy="epoch",        # FIX: was missing — required for eval to run
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,            # FIX: reduced from 5 to 3
    weight_decay=0.01,
    load_best_model_at_end=True,   # NEW: keep best checkpoint automatically
    metric_for_best_model="f1",    # NEW: save based on F1 score
    report_to="none",              # suppress wandb/tensorboard warnings
)


In [21]:

from transformers import Trainer

train_dataset = tokenized_datasets['train'].shuffle(seed=42).select(range(20000))
eval_dataset  = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)


In [22]:
ed.
print("Starting training...")
trainer.train()
print("Training complete!")


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.269630,0.291122,0.888000,0.885010,0.886831,0.883197
2,0.197449,0.372075,0.895000,0.890511,0.906582,0.875000
3,0.075879,0.510607,0.886000,0.882474,0.887967,0.877049


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training complete!


In [23]:
results = trainer.evaluate()
print("Evaluation Results:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


Evaluation Results:
  eval_loss: 0.3730
  eval_accuracy: 0.8950
  eval_f1: 0.8905
  eval_precision: 0.9066
  eval_recall: 0.8750
  eval_runtime: 7.7114
  eval_samples_per_second: 129.6780
  eval_steps_per_second: 8.1700
  epoch: 3.0000


In [24]:
model.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")
print("Model saved to ./sentiment_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./sentiment_model


In [25]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()  # NEW: set to eval mode — disables dropout for inference

test_texts = [
    "This movie was fantastic! I loved it.",
    "Terrible movie. Waste of time.",
    "It was okay, nothing special.",
    "Best film I've seen in years!",
    "I couldn't finish it, very boring."
]

inputs = tokenizer(test_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():  # NEW: no_grad for faster inference, saves memory
    outputs = model(**inputs)

predictions = np.argmax(outputs.logits.detach().cpu().numpy(), axis=1)

# NEW: Also show confidence scores
import torch.nn.functional as F
probs = F.softmax(outputs.logits, dim=1).detach().cpu().numpy()

print("\n=== Inference Results ===")
for text, pred, prob in zip(test_texts, predictions, probs):
    label = id2label[pred]
    confidence = prob[pred] * 100
    print(f"Text: {text}")
    print(f"  --> Sentiment: {label} (confidence: {confidence:.1f}%)\n")



=== Inference Results ===
Text: This movie was fantastic! I loved it.
  --> Sentiment: Positive (confidence: 99.7%)

Text: Terrible movie. Waste of time.
  --> Sentiment: Negative (confidence: 99.7%)

Text: It was okay, nothing special.
  --> Sentiment: Negative (confidence: 92.4%)

Text: Best film I've seen in years!
  --> Sentiment: Positive (confidence: 99.4%)

Text: I couldn't finish it, very boring.
  --> Sentiment: Negative (confidence: 99.2%)



In [20]:
from transformers import pipeline
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
classifier('very very very boring ')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9997970461845398}]